# Vector Index Benchmarking Framework — interactive demo

**Performance Evaluation of Vector Indexing Techniques for AI Search Systems**  
COMP.8157 Advanced Database Topics, 2026S — University of Windsor — **Group 4**

This notebook runs the complete framework in the cloud, with nothing installed locally.
It is the hosted entry point referred to in the D.4.4 Deployment document: **Runtime → Run all**
reproduces the project's results from a clean machine in roughly ten minutes.

What it does, in order:

1. clones the repository and installs the dependencies,
2. runs the scalability sweep (Flat vs IVF vs HNSW, 1k → 100k synthetic vectors),
3. runs the real Sentence-BERT benchmark over 20,000 ABC News headlines,
4. runs the IVF/HNSW parameter-tuning sweep,
5. queries the SQLite results database that every run writes into.

Source: [https://github.com/Devz0201/COMP8157-Vector-Indexing](https://github.com/Devz0201/COMP8157-Vector-Indexing)


---
## 1. Set up the environment

Colab already provides NumPy, pandas, and matplotlib. The two additions are FAISS (the index
library under test) and sentence-transformers (the embedding model from the proposal).


In [ ]:
!git clone --depth 1 \
    https://github.com/Devz0201/COMP8157-Vector-Indexing.git \
    2>/dev/null || echo 'already cloned'
%cd /content/COMP8157-Vector-Indexing
!pip install -q faiss-cpu PyYAML sentence-transformers
print()
import sys

import faiss

print('python', sys.version.split()[0], '| faiss', faiss.__version__)


Create the results database. Its schema lives in `db/schema.sql`; the runners apply it
automatically, so this step is only here to show what is being created.


In [ ]:
!python db_cli.py init


---
## 2. Scalability sweep — how the three index families behave as the data grows

Synthetic clustered vectors, so no download is needed and dataset size is a free parameter.
The expected result: Flat's latency grows linearly with N, IVF stays fast and holds recall,
and HNSW is the fastest of the three but loses recall as N grows unless `ef_search` is raised.


In [ ]:
!python run_benchmark.py --config configs/scalability.yaml --notes 'Colab demo'


In [ ]:
from IPython.display import Image, display

display(Image('results/scalability/scalability.png'))


---
## 3. Real-data benchmark — 20,000 news headlines, Sentence-BERT

`data/headlines.txt` ships with the repository, so this cell runs straight away. The first
run downloads the `all-MiniLM-L6-v2` model (~90 MB) and then embeds the corpus; on Colab's
CPU runtime that takes two to three minutes.

To rebuild the corpus from its source instead, run `!python fetch_dataset.py --n 20000`
(Harvard Dataverse, doi:10.7910/DVN/SYBGZL, CC0 — no account needed).


In [ ]:
!python run_benchmark.py --config configs/text_sbert.yaml --notes 'Colab demo'


In [ ]:
display(Image('results/sbert/comparison_bars.png'))


---
## 4. Parameter tuning — what the speed/accuracy knob actually costs

IVF's `nprobe` and HNSW's `ef_search` are both query-time settings, so each index is built
once and re-queried at every value in the sweep.


In [ ]:
!python tune_parameters.py --config configs/tuning.yaml --notes 'Colab demo'


In [ ]:
display(Image('results/tuning/tuning_tradeoff.png'))


---
## 5. The results database

Every run above also wrote itself into `db/vecbench_results.db`. That is what makes runs
comparable with each other rather than only readable one CSV at a time.

First, the runs recorded in this session:


In [ ]:
!python db_cli.py runs


Filtering: which configurations answered a query in under a millisecond *while still*
holding recall at or above 0.95? This is the question the framework exists to answer, and
it is one command.


In [ ]:
!python db_cli.py results --min-recall 0.95 --max-latency 1.0 --sort latency


The execution log of the most recent run, filtered — the same mechanism the User Guide
describes for troubleshooting a failed run.


In [ ]:
!python db_cli.py logs --run latest --level INFO --tail 15


A per-method rollup across everything run in this session:


In [ ]:
!python db_cli.py summary


Ad-hoc SQL, for anything the fixed commands do not cover:


In [ ]:
!python db_cli.py sql \
    "SELECT method, n_vectors, ROUND(latency_mean_ms,4) AS ms, recall_at_k \
     FROM v_scalability ORDER BY n_vectors, method"


---
## 6. Change something and re-run

Nothing about an experiment lives in the source code — it is all in the YAML config. The
cell below widens HNSW's graph search (`ef_search` 64 → 256) and re-runs the largest scale
only, which is how the scalability sweep's recall decay is recovered.


In [ ]:
import yaml

cfg = yaml.safe_load(open('configs/scalability.yaml'))
cfg['params']['hnsw']['ef_search'] = 256      # search wider at query time
cfg['scalability']['sizes'] = [100000]        # only the largest scale
cfg['outdir'] = 'results/colab_efsearch'
yaml.safe_dump(cfg, open('configs/colab_efsearch.yaml', 'w'), sort_keys=False)

!python run_benchmark.py --config configs/colab_efsearch.yaml --notes 'ef_search=256 at 100k'


Compare the two settings at 100,000 vectors side by side, straight out of the database:


In [ ]:
!python db_cli.py sql \
    "SELECT run_id, label, ROUND(latency_mean_ms,4) AS ms, recall_at_k \
     FROM measurement WHERE n_vectors=100000 AND method='HNSW' ORDER BY run_id"


---
## 7. Take the results with you

Export any filtered result set to CSV, or download the whole database file and open it in
any SQLite client.


In [ ]:
!python db_cli.py export results --out results/colab_export.csv

from google.colab import files

files.download('results/colab_export.csv')
# files.download('db/vecbench_results.db')   # uncomment for the full database


---
### Documentation

| Document | Where |
|---|---|
| D.4.1 User Requirements and Analysis | `docs/` |
| D.4.2 Design Document | `docs/` |
| D.4.4 Deployment Document | `docs/` |
| D.4.5 User Guide | `docs/` |
| Database schema | `db/schema.sql` |
| Dataset provenance and licence | `data/DATASET.md` |

**Group 4** — Devu Babu Sheeja · Ashwin Senthur Pandian · Rishi G. Patel · Nikhil Goud Nathi
